In [4]:
from ready_for_ML import FootballPreprocessor
from xgboost import XGBClassifier
import pandas as pd
from sklearn.metrics import classification_report


ModuleNotFoundError: No module named 'ready_for_ML'

In [ ]:
processor = FootballPreprocessor(
    "dataset/23-24.csv"
)


df = processor.load_data()


X, y = processor.prepare_features(df)


X_train, X_test, y_train, y_test = processor.split_data(
    X,
    y
)


# THIS WAS MISSING
X_train, X_test, y_train, y_test = processor.preprocess(
    X_train,
    X_test,
    y_train,
    y_test
)

xgb = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=4,
    random_state=42
)

xgb.fit(
    X_train,
    y_train
)

y_pred = xgb.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.58      0.52      0.55        27
           1       0.29      0.31      0.30        13
           2       0.68      0.72      0.70        32

    accuracy                           0.57        72
   macro avg       0.52      0.51      0.51        72
weighted avg       0.57      0.57      0.57        72



In [ ]:
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix

# --- Load your full merged dataset ---
all_df = pd.read_csv(r"C:\Users\misog\SCHOOL\Summer project\ML-football-odds\dataset\all_seasons.csv")

# --- Chronological season split ---
train_seasons = ["21-22", "22-23", "23-24", "24-25"]
test_season = "25-26"

train_df = all_df[all_df["Season"].isin(train_seasons)].copy()
test_df = all_df[all_df["Season"] == test_season].copy()

print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")

drop_cols = ["Date", "Time", "HomeTeam", "AwayTeam", "FTHG", "FTAG", "FTR", "Season"]

X_train = train_df.drop(columns=drop_cols)
y_train_raw = train_df["FTR"]

X_test = test_df.drop(columns=drop_cols)
y_test_raw = test_df["FTR"]

X_train = X_train.fillna(0)
X_test = X_test.fillna(0)

# --- Encode FTR (A/D/H) into 0/1/2 ---
le = LabelEncoder()
y_train = le.fit_transform(y_train_raw)
y_test = le.transform(y_test_raw)
print("Class mapping:", dict(zip(le.classes_, le.transform(le.classes_))))

# --- Train ---
xgb = XGBClassifier(
    n_estimators=300,
    max_depth=10,
    min_child_weight=5,   # XGBoost's equivalent of min_samples_leaf
    random_state=42,
    n_jobs=-1,
    eval_metric="mlogloss"
)
xgb.fit(X_train, y_train)

# --- Predict & evaluate ---
y_pred = xgb.predict(X_test)

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=le.classes_))

print("Confusion Matrix (rows=actual, cols=predicted, order = classes below):")
print(le.classes_)
print(confusion_matrix(y_test, y_pred))

Train shape: (1439, 49)
Test shape: (360, 49)
Class mapping: {'A': np.int64(0), 'D': np.int64(1), 'H': np.int64(2)}

Classification Report:
              precision    recall  f1-score   support

           A       0.40      0.41      0.41       109
           D       0.38      0.08      0.13        99
           H       0.48      0.72      0.58       152

    accuracy                           0.45       360
   macro avg       0.42      0.41      0.37       360
weighted avg       0.43      0.45      0.41       360

Confusion Matrix (rows=actual, cols=predicted, order = classes below):
['A' 'D' 'H']
[[ 45   7  57]
 [ 31   8  60]
 [ 36   6 110]]


In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score


class WalkForwardValidator:

    def __init__(self, seasons):
        self.seasons = seasons

    def split(self, meta):
        unique_seasons = sorted(meta["Season"].unique())

        for i in range(1, len(unique_seasons)):
            train_seasons = unique_seasons[:i]
            test_season = unique_seasons[i]

            train_idx = meta[meta["Season"].isin(train_seasons)].index
            test_idx = meta[meta["Season"] == test_season].index

            yield (train_idx, test_idx, train_seasons, test_season)


# --- Load your full merged dataset ---
all_df = pd.read_csv(r"C:\Users\misog\SCHOOL\Summer project\ML-football-odds\dataset\all_seasons.csv")
all_df = all_df.reset_index(drop=True)  # important: validator yields positional-style index, keep it clean

print(f"Full dataset shape: {all_df.shape}")
print(f"Seasons found: {sorted(all_df['Season'].unique())}")

drop_cols = ["Date", "Time", "HomeTeam", "AwayTeam", "FTHG", "FTAG", "FTR", "Season"]

validator = WalkForwardValidator(seasons=sorted(all_df["Season"].unique()))

fold_results = []
all_importances = []
class_labels = sorted(all_df["FTR"].unique())  # fixed label order across folds for aggregation

for fold_num, (train_idx, test_idx, train_seasons, test_season) in enumerate(validator.split(all_df), start=1):

    train_df = all_df.loc[train_idx]
    test_df = all_df.loc[test_idx]

    print(f"\n{'='*60}")
    print(f"Fold {fold_num}: train on {train_seasons} -> test on {test_season}")
    print(f"Train shape: {train_df.shape} | Test shape: {test_df.shape}")

    X_train = train_df.drop(columns=drop_cols)
    y_train = train_df["FTR"]

    X_test = test_df.drop(columns=drop_cols)
    y_test = test_df["FTR"]

    X_train = X_train.fillna(0)
    X_test = X_test.fillna(0)

    xgb = XGBClassifier(
    n_estimators=300,
    max_depth=10,
    min_child_weight=5,   # XGBoost's equivalent of min_samples_leaf
    random_state=42,
    n_jobs=-1,
    eval_metric="mlogloss"
)
    xgb.fit(X_train, y_train)

    y_pred = xgb.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    f1_macro = f1_score(y_test, y_pred, average="macro")

    print(f"Accuracy: {acc:.4f} | Macro F1: {f1_macro:.4f}")
    print(classification_report(y_test, y_pred, labels=class_labels))
    print("Confusion Matrix (rows=actual, cols=predicted):")
    print(class_labels)
    print(confusion_matrix(y_test, y_pred, labels=class_labels))

    fold_results.append({
        "fold": fold_num,
        "test_season": test_season,
        "train_seasons": train_seasons,
        "accuracy": acc,
        "f1_macro": f1_macro,
        "n_train": len(train_df),
        "n_test": len(test_df),
    })

    all_importances.append(pd.Series(xgb.feature_importances_, index=X_train.columns))

# --- Aggregate across folds ---
results_df = pd.DataFrame(fold_results)
print(f"\n{'='*60}")
print("WALK-FORWARD SUMMARY")
print(f"{'='*60}")
print(results_df[["fold", "test_season", "n_train", "n_test", "accuracy", "f1_macro"]])

print(f"\nMean accuracy across folds: {results_df['accuracy'].mean():.4f} (+/- {results_df['accuracy'].std():.4f})")
print(f"Mean macro F1 across folds: {results_df['f1_macro'].mean():.4f} (+/- {results_df['f1_macro'].std():.4f})")

# --- Aggregate feature importance across folds ---
importance_df = pd.concat(all_importances, axis=1)
importance_df.columns = [f"fold_{r['fold']}_{r['test_season']}" for r in fold_results]
mean_importance = importance_df.mean(axis=1).sort_values(ascending=False)

print("\nTop 15 features by mean importance across all folds:")
print(mean_importance.head(15))

Train shape: (5033, 49)
Test shape: (360, 49)
Class mapping: {'A': np.int64(0), 'D': np.int64(1), 'H': np.int64(2)}

Classification Report:
              precision    recall  f1-score   support

           A       0.45      0.39      0.42       109
           D       0.29      0.08      0.13        99
           H       0.48      0.76      0.59       152

    accuracy                           0.46       360
   macro avg       0.41      0.41      0.38       360
weighted avg       0.42      0.46      0.41       360

Confusion Matrix (rows=actual, cols=predicted, order = classes below):
['A' 'D' 'H']
[[ 42   8  59]
 [ 26   8  65]
 [ 25  12 115]]
Full dataset shape: (5393, 49)
Seasons found: ['11-12', '12-13', '13-14', '14-15', '15-16', '16-17', '17-18', '18-19', '19-20', '20-21', '21-22', '22-23', '23-24', '24-25', '25-26']

Fold 1: train on ['11-12'] -> test on 12-13
Train shape: (359, 49) | Test shape: (359, 49)


ValueError: Invalid classes inferred from unique values of `y`.  Expected: [0 1 2], got ['A' 'D' 'H']

In [ ]:
import pandas as pd
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# --- Same season-split logic as the single-run cell above, but regularized to
#     fix the overfitting we found (that config hit 100% train accuracy) ---
all_df = pd.read_csv(r"C:\Users\misog\SCHOOL\Summer project\ML-football-odds\dataset\all_seasons.csv")

train_seasons = ["21-22", "22-23", "23-24"]
val_season = "24-25"   # held out from training, used only for early stopping
test_season = "25-26"

train_df = all_df[all_df["Season"].isin(train_seasons)].copy()
val_df = all_df[all_df["Season"] == val_season].copy()
test_df = all_df[all_df["Season"] == test_season].copy()

print(f"Train shape: {train_df.shape}")
print(f"Val shape: {val_df.shape}")
print(f"Test shape: {test_df.shape}")

drop_cols = ["Date", "Time", "HomeTeam", "AwayTeam", "FTHG", "FTAG", "FTR", "Season"]

X_train = train_df.drop(columns=drop_cols).fillna(0)
y_train_raw = train_df["FTR"]

X_val = val_df.drop(columns=drop_cols).fillna(0)
y_val_raw = val_df["FTR"]

X_test = test_df.drop(columns=drop_cols).fillna(0)
y_test_raw = test_df["FTR"]

# --- Encode FTR (A/D/H) into 0/1/2 ---
le = LabelEncoder()
y_train = le.fit_transform(y_train_raw)
y_val = le.transform(y_val_raw)
y_test = le.transform(y_test_raw)
print("Class mapping:", dict(zip(le.classes_, le.transform(le.classes_))))

# --- Train, with the regularization the unconstrained version was missing ---
xgb = XGBClassifier(
    n_estimators=300,
    max_depth=4,             # was 10 -- way too deep for ~1400 training rows
    learning_rate=0.05,      # was left at the default (~0.3)
    subsample=0.8,           # row subsampling per tree, like RF's bootstrap
    colsample_bytree=0.8,    # feature subsampling per tree, like RF's random subset
    min_child_weight=5,
    random_state=42,
    n_jobs=-1,
    eval_metric="mlogloss",
    early_stopping_rounds=20,
)
xgb.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=False,
)

print(f"\nBest iteration: {xgb.best_iteration} (out of {xgb.n_estimators} max) -- early stopping cut off the rest")

# --- Predict & evaluate ---
y_pred = xgb.predict(X_test)

print(f"\nTrain accuracy: {accuracy_score(y_train, xgb.predict(X_train)):.4f}  (was 1.0000 unregularized)")
print(f"Test accuracy: {accuracy_score(y_test, y_pred):.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=le.classes_))

print("Confusion Matrix (rows=actual, cols=predicted, order = classes below):")
print(le.classes_)
print(confusion_matrix(y_test, y_pred))
